# Buổi G7: Thực hành tổng hợp Phần G, dự đoán tiến triển bệnh kèm khoảng tin cậy

**Hiện trường.** Phòng khám của chị Hạnh có dữ liệu 442 bệnh nhân tiểu đường với 10 chỉ số cơ bản (tuổi, BMI, huyết áp, các chỉ số máu). Bác sĩ muốn một mô hình đoán **mức tiến triển bệnh sau một năm** (một con số), để ưu tiên bệnh nhân theo dõi sát hơn. Hội đồng đặt ba câu hỏi: *mô hình nào là đủ tốt?* (có cần mạng nơ-ron không), *cột mới và chọn cột có giúp không?*, và *dự đoán này có thể lệch bao nhiêu?*

Đây là bài tổng hợp của Phần G: dùng **mạng nơ-ron** (G1, G2), **cột mới** (G4), **chọn cột** (G3) và **khoảng dự đoán** (G5), với kỷ luật của Phần D: pipeline, kiểm định chéo lặp, tập thi niêm phong mở một lần.

**Ẩn dụ:** như một cuộc thi tuyển chọn. Các thí sinh (Ridge, Ridge có cột mới, rừng tăng cường, mạng nơ-ron) thi vòng loại bằng kiểm định chéo. Thí sinh nào chỉ hơn sát nút thì chọn người **đơn giản hơn**. Chung kết (tập thi) chỉ diễn ra một lần.

**Mindset của người làm dữ liệu:** đáp án không phải "mô hình mạnh nhất" mà là **mô hình đơn giản nhất đủ tốt**, kèm lời nói thật về mức sai lệch.

**Lưu ý:** dữ liệu `load_diabetes` của sklearn dùng để học, không dùng ra quyết định y tế thật.

## Bản đồ các chặng

| Chặng | Câu hỏi của chặng |
|---|---|
| Chặng 0 | Dựng mốc: tập thi niêm phong, mô hình đoán trung bình, Ridge |
| Chặng 1 | Cột mới và chọn cột, nằm trong pipeline |
| Chặng 2 | Mạng nơ-ron, rừng tăng cường: cùng thi một sân |
| Chặng 3 | Chọn người thắng cuộc theo quy tắc một sai số chuẩn |
| Chặng 4 | Khoảng dự đoán bằng conformal, kiểm theo vùng |
| Chặng 5 | Mở tập thi một lần và viết báo cáo |

## Chặng 0: Dựng mốc

**Câu hỏi:** trước khi thử gì, sân thi và mốc so sánh là gì?

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
BLUE, GRAY = "#1F4E79", "#8C8C8C"
plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

**Giải thích từng dòng**

- `import numpy as np`: mượn thư viện numpy (máy tính bỏ túi cho cả dãy số), đặt tên ngắn là `np` để gõ cho nhanh.
- `import pandas as pd`: mượn pandas, thư viện làm việc với bảng số liệu giống Excel, tên ngắn là `pd`.
- `import matplotlib.pyplot as plt`: mượn công cụ vẽ biểu đồ, tên ngắn là `plt`.
- `BLUE, GRAY = "#1F4E79", "#8C8C8C"`: gán hai màu dùng suốt bài (xanh đậm và xám). Một dòng gán cùng lúc hai biến, mỗi biến một mã màu.
- `plt.rcParams["figure.figsize"] = (6, 4)`: đặt cỡ hình mặc định: rộng 6, cao 4 (đơn vị inch).
- `plt.rcParams["axes.spines.top"] = False`: bỏ đường viền phía trên khung hình cho gọn mắt.
- `plt.rcParams["axes.spines.right"] = False`: bỏ luôn đường viền bên phải.

In [2]:
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, RepeatedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import RidgeCV
from sklearn.dummy import DummyRegressor
X_all, y_all = load_diabetes(return_X_y=True, as_frame=True)
X_rest, X_test, y_rest, y_test = train_test_split(X_all, y_all, test_size=0.25, random_state=0)
X_train, X_cal, y_train, y_cal = train_test_split(X_rest, y_rest, test_size=0.3, random_state=1)
print("train", len(X_train), "| calibration", len(X_cal), "| sealed test", len(X_test))
cv = RepeatedKFold(n_splits=5, n_repeats=3, random_state=0)
def cv_scores(model, X=None, y=None):
    X = X_train if X is None else X; y = y_train if y is None else y
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        return cross_val_score(model, X, y, cv=cv, scoring="r2")
dummy_s = cv_scores(DummyRegressor())
ridge = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 20)))
ridge_s = cv_scores(ridge)
print("dummy (predict the mean) R2:", round(dummy_s.mean(), 3), "| Ridge R2:", round(ridge_s.mean(), 3), "+/-", round(ridge_s.std() / np.sqrt(len(ridge_s)), 3))

train 231 | calibration 100 | sealed test 111


dummy (predict the mean) R2: -0.048 | Ridge R2: 0.476 +/- 0.02


**Giải thích từng dòng**

- `train_test_split(...) hai lần`: chia ba phần: **tập thi niêm phong** (chỉ mở ở chặng 5), **tập hiệu chỉnh** (cho khoảng dự đoán, chặng 4) và **tập học** (để so sánh và huấn luyện).
- `RepeatedKFold(n_splits=5, n_repeats=3)`: kiểm định chéo 5 phần lặp 3 lần với cách chia khác nhau: 15 điểm cho mỗi mô hình, ổn định hơn một lần chạy (D1).
- `def cv_scores(model, ...)`: hàm chung để mọi thí sinh thi **cùng cách chia**, tắt cảnh báo hội tụ của mạng nơ-ron.
- `DummyRegressor()`: mô hình ngốc nhất: luôn đoán trung bình tập học. Mốc dưới cùng (C1).
- `RidgeCV(alphas=np.logspace(-2, 3, 20))`: Ridge tự chọn phanh `alpha` bằng kiểm định chéo nội bộ (B6, D4).

**Cách đọc.**
- **Con số:** mô hình đoán trung bình có R² **-0,048** (quanh 0 như mong đợi); Ridge **0,476** (sai số chuẩn **0,020**).
- **Bức tranh:** bài này khó: ngay cả mô hình tốt cũng chỉ giải thích được khoảng một nửa độ biến thiên của tiến triển bệnh, phần còn lại là yếu tố ngoài 10 chỉ số. Mốc Ridge là thứ mọi mô hình phức tạp phải vượt.
- **Câu hỏi tiếp:** cột mới và chọn cột có giúp mốc đó không?

**Mang về chặng 1:** chốt tập thi niêm phong, mốc và cách chia trước khi thử.

## Chặng 1: Cột mới và chọn cột

**Câu hỏi:** thêm tích và bình phương các cột (G4) rồi chọn lọc lại (G3), có hơn Ridge thuần không?

In [3]:
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import Ridge
rows = {}
rows["Ridge (10 columns)"] = ridge_s
poly_all = make_pipeline(PolynomialFeatures(2, include_bias=False), StandardScaler(), RidgeCV(alphas=np.logspace(-1, 4, 20)))
rows["Ridge + poly(2), all 65 columns"] = cv_scores(poly_all)
for k in (10, 20, 30):
    p = make_pipeline(PolynomialFeatures(2, include_bias=False), StandardScaler(), SelectKBest(f_regression, k=k), RidgeCV(alphas=np.logspace(-1, 4, 20)))
    rows[f"Ridge + poly(2) + keep {k}"] = cv_scores(p)
fe_table = pd.DataFrame({k: {"R2": v.mean(), "se": v.std() / np.sqrt(len(v))} for k, v in rows.items()}).T.round(3)
print(fe_table.to_string())

                                    R2     se
Ridge (10 columns)               0.476  0.020
Ridge + poly(2), all 65 columns  0.435  0.020
Ridge + poly(2) + keep 10        0.456  0.019
Ridge + poly(2) + keep 20        0.433  0.019
Ridge + poly(2) + keep 30        0.420  0.019


**Giải thích từng dòng**

- `PolynomialFeatures(2, include_bias=False)`: từ 10 cột sinh ra 65 cột: 10 cột gốc, 10 bình phương, 45 tích đôi một (G4).
- `SelectKBest(f_regression, k=k)`: chọn lại `k` cột trong 65 cột, **nằm trong pipeline** nên được học lại ở từng phần kiểm định chéo (G3, chặng 4: tránh rò rỉ).
- `RidgeCV(...)`: Ridge tự chọn phanh: nhiều cột mới thì cần phanh mạnh hơn (G4, chặng 5).

**Cách đọc.**
- **Con số:** Ridge thuần R² **0,476**; thêm toàn bộ 65 cột **0,435**; thêm rồi giữ 20 cột **0,433**. Sai số chuẩn cỡ **0,019**.
- **Bức tranh:** các biến thể có cột mới hơn kém Ridge thuần không quá một sai số chuẩn, tức là không đủ bằng chứng rằng chúng tốt hơn. Dữ liệu chỉ có 231 dòng học; thêm 55 cột mới là thêm nhiều núm vặn trên ít dữ liệu (G4, chặng 5).
- **Quy tắc đọc:** cột mới đáng giữ khi thắng rõ **vượt sai số chuẩn**. Nếu không thì giữ phiên bản đơn giản.

**Mang về chặng 2:** cột mới phải thắng rõ mới đáng giữ.

## Chặng 2: Mạng nơ-ron và rừng tăng cường

**Câu hỏi:** mô hình phức tạp có đáng không trên bộ dữ liệu nhỏ này?

In [4]:
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.compose import TransformedTargetRegressor
def scaled_mlp(hidden, alpha):
    net = make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=hidden, alpha=alpha, solver='lbfgs', max_iter=500, random_state=0))
    return TransformedTargetRegressor(regressor=net, transformer=StandardScaler())
cands = {
    "ridge": ridge,
    "ridge + poly(2) + keep 20": make_pipeline(PolynomialFeatures(2, include_bias=False), StandardScaler(), SelectKBest(f_regression, k=20), RidgeCV(alphas=np.logspace(-1, 4, 20))),
    "hist gradient boosting": HistGradientBoostingRegressor(max_depth=3, learning_rate=0.05, max_iter=150, random_state=0),
    "MLP (16), alpha=1": scaled_mlp((16,), 1.0),
    "MLP (16), alpha=10": scaled_mlp((16,), 10.0),
    "MLP (64, 64), alpha=0.01": scaled_mlp((64, 64), 0.01),
}
scores = {name: cv_scores(m) for name, m in cands.items()}
board = pd.DataFrame({n: {"CV R2": s.mean(), "se": s.std() / np.sqrt(len(s))} for n, s in scores.items()}).T.round(3).sort_values("CV R2", ascending=False)
print(board.to_string())

                           CV R2     se
ridge                      0.476  0.020
MLP (16), alpha=10         0.475  0.020
hist gradient boosting     0.451  0.025
ridge + poly(2) + keep 20  0.433  0.019
MLP (16), alpha=1          0.232  0.038
MLP (64, 64), alpha=0.01  -0.052  0.048


**Giải thích từng dòng**

- `HistGradientBoostingRegressor(max_depth=3, ...)`: cây tăng cường nhanh (E4) với cây nông và học chậm vì dữ liệu nhỏ.
- `MLPRegressor(hidden_layer_sizes=(16,), alpha=1.0, solver="lbfgs")`: mạng nơ-ron hồi quy 1 tầng ẩn 16 nơ-ron, phanh `alpha` vừa phải, bộ chỉnh `lbfgs` cho dữ liệu nhỏ (G1, G2).
- `MLPRegressor(hidden_layer_sizes=(64, 64), alpha=0.01)`: mạng lớn, gần như không phanh: để xem học thừa trên dữ liệu nhỏ ra sao.
- `scores = {name: cv_scores(m) ...}`: mọi thí sinh thi trên **cùng 15 phần chia**, nên so sánh công bằng.

**Cách đọc.**
- **Con số:** bảng xếp hạng: **ridge** đứng đầu với R² **0,476** (sai số chuẩn 0,020); mạng lớn gần không phanh đạt **-0,052**.
- **Bức tranh:** so các mô hình theo bảng, đọc khoảng cách bằng **sai số chuẩn** (cột `se`): hai mô hình cách nhau dưới một đến hai sai số chuẩn thì chưa phân biệt được ai hơn. Mạng nơ-ron trên chỉ 231 dòng học rất dễ học thừa (G2, chặng 5); mạng càng lớn và phanh càng ít thì càng tệ, mạng nhỏ có phanh mạnh thì khá hơn. Với dữ liệu nhỏ, mô hình đơn giản thường ngang hoặc hơn mô hình phức tạp.
- **Câu hỏi tiếp:** khi nhiều mô hình sát nút, chọn ai?

**Mang về chặng 3:** mạng nơ-ron cần phanh và chuẩn hóa cả đích; trên dữ liệu nhỏ mô hình đơn giản thường ngang hoặc hơn.

## Chặng 3: Chọn người thắng cuộc

**Câu hỏi:** nếu hai mô hình sát nút, chọn mô hình nào?

In [5]:
order = ["ridge", "ridge + poly(2) + keep 20", "MLP (16), alpha=10", "MLP (16), alpha=1", "hist gradient boosting", "MLP (64, 64), alpha=0.01"]
best_name = board.index[0]
threshold = board.loc[best_name, "CV R2"] - board.loc[best_name, "se"]
chosen = next(n for n in order if board.loc[n, "CV R2"] >= threshold)
print("best by CV:", best_name, "| one-SE floor:", round(threshold, 3))
print("candidates inside the one-SE band:", [n for n in order if board.loc[n, "CV R2"] >= threshold])
print("chosen (simplest inside the band):", chosen)
gap = scores[chosen] - scores[best_name]
print("paired difference chosen - best (per split): mean", round(gap.mean(), 4), "| se", round(gap.std() / np.sqrt(len(gap)), 4))

best by CV: ridge | one-SE floor: 0.456
candidates inside the one-SE band: ['ridge', 'MLP (16), alpha=10']
chosen (simplest inside the band): ridge
paired difference chosen - best (per split): mean 0.0 | se 0.0


**Giải thích từng dòng**

- `order = [...]`: thứ tự **từ đơn giản đến phức tạp** do ta quy ước: Ridge đơn giản nhất, mạng lớn phức tạp nhất.
- `threshold = best - se`: **quy tắc một sai số chuẩn** (D4): mọi mô hình có điểm không thấp hơn điểm tốt nhất quá một sai số chuẩn đều coi là ngang nhau.
- `next(n for n in order if ...)`: trong số ngang nhau, chọn mô hình **đầu tiên trong thứ tự đơn giản**.
- `scores[chosen] - scores[best_name]`: hiệu cặp trên từng phần chia: nhìn cả hai mô hình trên cùng dữ liệu nên chính xác hơn so hai trung bình.

**Cách đọc.**
- **Con số:** tốt nhất theo kiểm định chéo là **ridge**; ngưỡng một sai số chuẩn là **0,456**; mô hình được chọn là **ridge**. Hiệu cặp (được chọn trừ tốt nhất): **0,0000** ± 0,0000.
- **Bức tranh:** trong nhóm sát nút, ta chọn mô hình đơn giản nhất: dễ giải thích với bác sĩ, ít núm vặn, ít rủi ro học thừa. Nếu mô hình tốt nhất thật sự vượt trội thì nó nằm một mình trên ngưỡng và quy tắc tự chọn nó.
- **Quy tắc đọc:** đây là cách ra quyết định có kỷ luật thay cho "chọn mô hình có điểm cao nhất theo trực giác".

**Mang về chặng 4:** trong nhóm sát nút, chọn mô hình đơn giản nhất.

## Chặng 4: Khoảng dự đoán bằng conformal

**Câu hỏi:** mô hình đã chọn, dự đoán có thể lệch bao nhiêu?

In [6]:
from sklearn.base import clone
final = clone(cands[chosen])
with warnings.catch_warnings():
    warnings.simplefilter("ignore", ConvergenceWarning)
    final.fit(X_train, y_train)
alpha = 0.2
resid_cal = np.abs(y_cal - final.predict(X_cal))
k = int(np.ceil((len(resid_cal) + 1) * (1 - alpha)))
half_width = float(np.sort(resid_cal)[min(k, len(resid_cal)) - 1])
print("calibration rows:", len(resid_cal), "| half-width for a", int((1 - alpha) * 100), "% interval:", round(half_width, 1), "(target units)")
print("for reference: std of the target:", round(float(y_train.std()), 1))

calibration rows: 100 | half-width for a 80 % interval: 68.0 (target units)
for reference: std of the target: 77.6


**Giải thích từng dòng**

- `clone(cands[chosen])`: tạo bản sao chưa học của mô hình được chọn để huấn luyện lại sạch sẽ.
- `np.abs(y_cal - final.predict(X_cal))`: sai số tuyệt đối của mô hình trên **tập hiệu chỉnh** (chưa từng dùng để học).
- `np.ceil((n + 1) * (1 - alpha))`: vị trí phân vị có hiệu chỉnh cỡ mẫu nhỏ (G5, chặng 4).
- `half_width`: nửa bề rộng khoảng: dự đoán cộng trừ con số này sẽ phủ khoảng 80% số lần, nếu dữ liệu mới giống dữ liệu hiệu chỉnh.

**Cách đọc.**
- **Con số:** nửa bề rộng khoảng 80% là **68,0** đơn vị tiến triển bệnh; độ lệch chuẩn của chính đại lượng cần đoán là **77,6**.
- **Bức tranh:** khoảng cộng trừ 68 so với độ trải của dữ liệu cho thấy dự đoán chỉ **thu hẹp phần nào** sự bất định, không xóa được. Đây là điều cần nói thẳng với bác sĩ.
- **Lưu ý:** conformal đơn giản này cho khoảng cùng bề rộng cho mọi bệnh nhân; nếu mức lệch khác nhau giữa các nhóm thì cần kiểm theo vùng (G5, chặng 2).
- **Câu hỏi tiếp:** khoảng này thật sự phủ 80% trên dữ liệu thi không?

**Mang về chặng 5:** khoảng dự đoán lấy từ sai số trên tập hiệu chỉnh riêng.

## Chặng 5: Mở tập thi một lần và viết báo cáo

**Câu hỏi:** trên dữ liệu thi niêm phong, mô hình và khoảng dự đoán thật sự thế nào?

In [7]:
pred_test = final.predict(X_test)
r2_test = 1 - ((y_test - pred_test) ** 2).sum() / ((y_test - y_test.mean()) ** 2).sum()
mae_test = float(np.mean(np.abs(y_test - pred_test)))
mae_dummy = float(np.mean(np.abs(y_test - y_train.mean())))
inside = np.abs(y_test - pred_test) <= half_width
level = pd.qcut(pred_test, 3, labels=["low predicted", "mid predicted", "high predicted"])
cov_by_level = pd.Series(np.asarray(inside)).groupby(level, observed=True).mean().round(3)
print("test R2:", round(float(r2_test), 3), "| MAE:", round(mae_test, 1), "| MAE of always-the-mean:", round(mae_dummy, 1))
print("interval coverage on test:", round(float(inside.mean()), 3), "(target 0.8, test rows:", len(y_test), ")")
print("coverage by predicted level:", cov_by_level.to_dict())

test R2: 0.374 | MAE: 44.8 | MAE of always-the-mean: 58.5
interval coverage on test: 0.793 (target 0.8, test rows: 111 )
coverage by predicted level: {'low predicted': 0.892, 'mid predicted': 0.811, 'high predicted': 0.676}


**Giải thích từng dòng**

- `final.predict(X_test)`: lần đầu và duy nhất mô hình thấy tập thi niêm phong.
- `mae_dummy`: sai số của mô hình ngốc (luôn đoán trung bình), để biết mô hình tốt hơn bao nhiêu.
- `np.abs(y_test - pred_test) <= half_width`: bệnh nhân nào có thực tế nằm trong khoảng dự đoán.
- `pd.qcut(pred_test, 3, ...)`: chia bệnh nhân thành 3 nhóm theo mức dự đoán thấp, vừa, cao để kiểm độ phủ theo vùng.

**Cách đọc.**
- **Con số:** trên tập thi niêm phong, mô hình **ridge** đạt R² **0,374**, sai số tuyệt đối trung bình **44,8** (mô hình ngốc: **58,5**). Khoảng 80% phủ **79,3%** trên **111** bệnh nhân thi; theo mức dự đoán: **{'low predicted': 0.892, 'mid predicted': 0.811, 'high predicted': 0.676}**.
- **Bức tranh:** điểm thi **0,374** so với điểm kiểm định chéo **0,476** (chênh **0,102**). Tập thi chỉ có **111** bệnh nhân nên R² của nó dao động đáng kể giữa các lần chia; chênh lệch cỡ này chưa phải bằng chứng rò rỉ, nhưng cũng nhắc ta rằng một con số thi duy nhất không chính xác tuyệt đối. Độ phủ theo mức dự đoán dao động từ **68%** đến **89%**: khoảng cố định phủ không đều giữa các nhóm bệnh nhân (đúng như cảnh báo ở G5), nhóm có dự đoán cao hay bị thiếu.
- **Câu trả lời cho hội đồng:** (1) **cần mạng nơ-ron không**: không cần; trên bài này mô hình đơn giản đã đủ và mạng không vượt được nó (xem bảng chặng 2); (2) **cột mới và chọn cột có giúp không**: xem chặng 1, chỉ giữ nếu thắng vượt một sai số chuẩn; (3) **lệch bao nhiêu**: khoảng cộng trừ **68** cho khoảng 80%, xem độ phủ theo vùng ở trên.

**Mang về:** mốc, thi cùng sân, chọn mô hình đơn giản trong nhóm sát nút, khoảng dự đoán kiểm theo vùng, mở tập thi một lần.

## Tổng kết

**Mang về theo từng chặng**

| Chặng | Mang về |
|---|---|
| 0 | Chốt tập thi niêm phong, mốc đơn giản và cách chia trước khi thử. |
| 1 | Cột mới và chọn cột đáng giữ khi thắng vượt một sai số chuẩn; nằm trong pipeline. |
| 2 | Mạng nơ-ron và rừng tăng cường cạnh tranh ngang ngửa mô hình đơn giản trên dữ liệu nhỏ; mạng lớn không phanh thua. |
| 3 | Nhóm sát nút: chọn mô hình đơn giản nhất theo quy tắc một sai số chuẩn. |
| 4 | Khoảng dự đoán bằng conformal trên tập hiệu chỉnh riêng. |
| 5 | Mở tập thi một lần; báo cáo mô hình, sai số, độ phủ khoảng, và nói thẳng giới hạn. |

**Bảng đối chiếu: đời thường, thuật ngữ, tên trong code**

| Đời thường | Thuật ngữ | Tên trong code |
|---|---|---|
| Tập thi niêm phong | tập kiểm tra cuối | `X_test`, `y_test` |
| Thi cùng sân cùng luật | kiểm định chéo lặp | `RepeatedKFold` |
| Thí sinh nhỏ gọn nhất trong nhóm ngang nhau | quy tắc một sai số chuẩn | `threshold = best - se` |
| Nói thật mức lệch | khoảng dự đoán | `half_width` |
| Có thật phủ 80%? | độ phủ | `inside.mean()` |
| Ngốc nhất làm mốc | mô hình cơ sở | `DummyRegressor` |

**Thử thách tự làm**

1. Thay `RepeatedKFold` bằng `n_repeats=10`. Bảng xếp hạng ở chặng 2 có ổn định hơn không?
2. Dùng `GradientBoostingRegressor(loss='quantile')` (G5) để làm khoảng dự đoán thay cho conformal đơn giản. Độ phủ theo vùng thế nào?
3. Chạy lại toàn bộ với `random_state` khác cho lần chia ban đầu. Mô hình thắng cuộc có đổi không?
4. Viết một đoạn 5 câu cho bác sĩ giải thích mô hình, sai số trung bình, và mức dự đoán đáng tin tới đâu.

**Checklist năng lực** (tự tick khi làm được)

- [ ] Chốt tập thi, mốc, cách chia trước khi thử.
- [ ] Đặt mọi bước học trong pipeline.
- [ ] So mô hình trên cùng các phần chia, đọc sai số chuẩn.
- [ ] Chọn mô hình đơn giản nhất trong nhóm sát nút.
- [ ] Làm khoảng dự đoán, kiểm độ phủ và nói thẳng giới hạn.

## Phần thưởng (Tip pro): Sang Phần H: triển khai và dự án


Mô hình tốt mới chỉ là nửa đường. Phần H dạy đưa mô hình ra dùng thật: lưu và nạp mô hình, giám sát khi dữ liệu đổi, và các dự án cuối khóa.